# PMRT component check: plain CNN baseline without the framework (Kaggle or Colab)

One notebook for everything:
1. trains a plain CNN that has none of the framework (`ABLATE`), with exactly the settings of the final run (same data, same folds, same multi-task loss, optimiser, EMA and early stopping; resumable),
2. compares test metrics of the final model with this baseline (same subjects, same metric code),
3. draws confusion matrices and ROC curves.

`ABLATE = "plaincnn"`: both signals (EEG, EMG) are stacked as a 2-channel input of one plain CNN: four Conv1d layers (kernels 7, 5, 3, 3, stride 2, 64-128-256-512 channels) with BatchNorm and ReLU, global average pooling and one small MLP head per target (HR, RMSSD, disorder, sleep stage). No Phy-SE or residual blocks, no Cross-Modal Fusion, no FPN, no multi-scale heads, no temporal Transformer; every 5-s window is predicted on its own.
`ABLATE = "plaincnn_ctx"`: the same plain CNN plus the temporal Transformer of the final model over 12 consecutive windows (60 s); the heads receive [CNN features, temporal context]. Comparing the three models shows what the plain CNN already achieves, what the 60-s context adds and what the rest of the framework adds.

**Kaggle:** Accelerator GPU T4, Internet On, then Save Version > Save & Run All (Commit). The data zip sharing must be "Anyone with the link" while it downloads. Steps 2-3 need the finished run `pmrt_mt3_cv5` (test predictions). If it is not found, the notebook trains the baseline, saves `ablation_run_<ABLATE>.zip` and skips 2-3: download that zip, unzip it into `sleep_project/runs/` on Google Drive and run this notebook in Colab (training is skipped there, only steps 2-3 run).
**Colab:** needs `sleep_project` on Google Drive with `runs/pmrt_mt3_cv5`. Upload `ablation_run_<ABLATE>.zip` into `sleep_project` (next to `folds_5.json`); the notebook unzips it into `runs/` by itself. Runtime: GPU.

In [ ]:
# Environment: Kaggle (data downloaded from the shared Drive zip) or Google Colab (data on Google Drive)
import os, sys, glob, shutil, subprocess, zipfile, time
from pathlib import Path
KAGGLE = os.path.exists("/kaggle/working")
FILE_ID = "11VPVORnZ951jWSle1wMWC0DAuLz5aFC-"
T_START, TIME_LIMIT_H = time.time(), 10.5
if KAGGLE:
    ROOT = Path("/kaggle/working")
    hits = sorted(glob.glob("/kaggle/input/**/x.npy", recursive=True))
    if not hits:
        tmp = Path("/tmp/capdata")
        if not list(tmp.rglob("x.npy")):
            try:
                import gdown
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True); import gdown
            tmp.mkdir(parents=True, exist_ok=True)
            gdown.download(id=FILE_ID, output="/tmp/cap.zip", quiet=False)
            zipfile.ZipFile("/tmp/cap.zip").extractall(tmp); os.remove("/tmp/cap.zip")
        hits = sorted(str(p) for p in tmp.rglob("x.npy"))
    assert hits, "x.npy not found in the zip / Kaggle input"
    DATA = Path(hits[0]).parent
    folds = sorted(glob.glob(str(DATA / "folds_5.json")) + glob.glob("/tmp/capdata/**/folds_5.json", recursive=True)
                   + glob.glob("/kaggle/input/**/folds_5.json", recursive=True))
    assert folds, "folds_5.json not found"
    FOLDS_FILE = Path(folds[0])
    for prev in glob.glob("/kaggle/input/**/pmrt_mt3_no_*_cv5", recursive=True):
        shutil.copytree(prev, ROOT / "runs" / Path(prev).name, dirs_exist_ok=True)
        print("previous Kaggle output found and copied:", prev)
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    ROOT = Path("/content/drive/MyDrive/sleep_project")
    DATA, FOLDS_FILE = ROOT / "cap_dataset_cache", ROOT / "folds_5.json"
    for name in ("physe", "stdse", "fusion", "singleconv", "oneconv", "plaincnn", "plaincnn_ctx"):
        z = ROOT / f"ablation_run_{name}.zip"
        if z.exists() and not (ROOT / "runs" / f"pmrt_mt3_no_{name}_cv5" / "fold4" / "best_dis.pt").exists():
            zipfile.ZipFile(z).extractall(ROOT / "runs")
            print("ablated model unzipped from", z)
print("platform:", "Kaggle" if KAGGLE else "Colab", "| data:", DATA, "| folds:", FOLDS_FILE, "| outputs:", ROOT)
print("files in data folder:", sorted(os.listdir(DATA)))

In [ ]:
# Model module of the final run (pmrt_mt3.py), written next to the notebook
import base64
from pathlib import Path
Path(ROOT, "pmrt_mt3.py").write_bytes(base64.b64decode("..."))
print("pmrt_mt3.py written to", ROOT)


In [ ]:
# Settings (training settings identical to the final run pmrt_mt3_cv5) (ABLATE: physe = plain ResNet block, stdse = standard SE-ResNet block, fusion = concat + 1x1 conv, singleconv = one convolution per stage, oneconv = one convolution layer per branch, plaincnn = plain CNN without the framework, plaincnn_ctx = plain CNN + temporal Transformer)
ABLATE = "plaincnn_ctx"
FOLDS = [0, 1, 2, 3, 4]
N_FOLDS = 5
SEQ_LEN = 12
EPOCHS, PATIENCE, MIN_EPOCHS = 60, 12, 15
BATCH_SEQ, LR, WD = 24, 5e-4, 0.05
PER_SUBJECT = 1500
EMA_DECAY = 0.999
SEED = 42
FEATURE_FRACTION = 0.2
MAX_POINTS_PER_CLASS = 1500


In [ ]:
# Libraries, device, paths and figure style
import json, importlib, warnings, shutil
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT))
import pmrt_mt3 as M
importlib.reload(M)
assert ABLATE in ("physe", "stdse", "fusion", "singleconv", "oneconv", "plaincnn", "plaincnn_ctx")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "- WARNING: no GPU")
LABEL = {None: "With module (final model)", "physe": "Without Phy-SE (plain ResNet block)", "stdse": "Phy-SE replaced by standard SE-ResNet block", "fusion": "Without Cross-Modal Fusion (concat + 1x1 conv)",
         "singleconv": "Single convolution per stage (no ResNet block)",
         "oneconv": "One convolution layer per branch (stem only, fixed pooling after it)",
         "plaincnn": "Plain CNN, 2-channel input (no fusion, FPN or Transformer)",
         "plaincnn_ctx": "Plain CNN + temporal Transformer (60 s context)"}
ABL = ROOT / "runs" / f"pmrt_mt3_no_{ABLATE}_cv5"
OUT = ROOT / f"ablation_analysis_{ABLATE}"
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir(parents=True)
cands = [ROOT / "runs" / "pmrt_mt3_cv5", Path(DATA) / "runs" / "pmrt_mt3_cv5"] + [Path(p) for p in glob.glob("/kaggle/input/**/pmrt_mt3_cv5", recursive=True) + glob.glob("/tmp/capdata/**/pmrt_mt3_cv5", recursive=True)]
def _ok(c, ck=False): return all((c / f"fold{k}" / n).exists() for k in FOLDS for n in (("test_pred.csv", "best_stress.pt", "best_dis.pt") if ck else ("test_pred.csv",)))
MAIN = next((c for c in cands if _ok(c, True)), None) or next((c for c in cands if _ok(c)), cands[0])
if not _ok(ABL):
    for p in glob.glob("/kaggle/input/**/pmrt_mt3_no_" + ABLATE + "_cv5", recursive=True) + glob.glob("/tmp/capdata/**/pmrt_mt3_no_" + ABLATE + "_cv5", recursive=True):
        if _ok(Path(p)): shutil.copytree(p, ABL, dirs_exist_ok=True); print("ablated model restored from", p); break
print("final run folder:", MAIN, "| predictions:", _ok(MAIN), "| checkpoints:", _ok(MAIN, True), "| ablated model complete:", _ok(ABL), "| ablated checkpoints:", _ok(ABL, True))
NAVY, SLATE = "#1d4e89", "#8c99a6"
CLS = ["#1d4e89", "#3f8f8a", "#d39a4f", "#9b8fae", "#b5546b", "#6f8f7e"]
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight", "font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})


In [ ]:
# Ablated model variants (same tensor shapes as the original model)
class PlainBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=2):
        super().__init__()
        self.main = nn.Sequential(nn.Conv1d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm1d(c_out), nn.ReLU(inplace=True),
                                  nn.Conv1d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm1d(c_out))
        self.short = nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm1d(c_out))
        self.act = nn.ReLU(inplace=True)
    def forward(self, x): return self.act(self.main(x) + self.short(x))

class PlainResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 512)):
        super().__init__()
        self.stem = M.StemBlock(widths[0])
        c, st = widths[0], []
        for w in widths: st.append(PlainBlock(c, w)); c = w
        self.stages = nn.ModuleList(st)
    def forward(self, x):
        x, feats = self.stem(x), []
        for s in self.stages: x = s(x); feats.append(x)
        return feats

class StdSEBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=2, r=16):
        super().__init__()
        self.main = nn.Sequential(nn.Conv1d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm1d(c_out), nn.ReLU(inplace=True),
                                  nn.Conv1d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm1d(c_out))
        h = max(c_out // r, 4)
        self.se1, self.se2 = nn.Linear(c_out, h), nn.Linear(h, c_out)
        self.short = nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm1d(c_out))
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        fm = self.main(x)
        s = torch.sigmoid(self.se2(F.relu(self.se1(fm.mean(-1))))).unsqueeze(-1)
        return self.act(fm * s + self.short(x))

class StdSEResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 512)):
        super().__init__()
        self.stem = M.StemBlock(widths[0])
        c, st = widths[0], []
        for w in widths: st.append(StdSEBlock(c, w)); c = w
        self.stages = nn.ModuleList(st)
    def forward(self, x):
        x, feats = self.stem(x), []
        for s in self.stages: x = s(x); feats.append(x)
        return feats

class SingleConvNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 512)):
        super().__init__()
        self.stem = M.StemBlock(widths[0])
        c, st = widths[0], []
        for w in widths:
            st.append(nn.Sequential(nn.Conv1d(c, w, 3, 2, 1, bias=False), nn.BatchNorm1d(w), nn.ReLU(inplace=True))); c = w
        self.stages = nn.ModuleList(st)
    def forward(self, x):
        x, feats = self.stem(x), []
        for s in self.stages: x = s(x); feats.append(x)
        return feats

class OneConvNet(nn.Module):
    def __init__(self, width=64):
        super().__init__()
        self.stem = M.StemBlock(width)
        self.pool = nn.AvgPool1d(3, 2, 1)
    def forward(self, x):
        x, feats = self.stem(x), []
        for _ in range(4):
            x = self.pool(x); feats.append(x)
        return feats

class PlainCNN(nn.Module):
    def __init__(self, context=False, widths=(64, 128, 256, 512), kernels=(7, 5, 3, 3), p=0.3):
        super().__init__()
        c, layers = 2, []
        for w, k in zip(widths, kernels):
            layers += [nn.Conv1d(c, w, k, 2, k // 2, bias=False), nn.BatchNorm1d(w), nn.ReLU(inplace=True)]; c = w
        self.conv = nn.Sequential(*layers)
        d = widths[-1]
        self.context = None
        if context:
            self.to_ctx = nn.Linear(d, 384)
            self.context = M.TemporalContext(128, SEQ_LEN)
            d += 128
        outs = {"hr": 1, "rmssd": 1, "dis": len(M.DISORDERS), "stg": len(M.STAGES)}
        self.pred = nn.ModuleDict({k: nn.Sequential(nn.Linear(d, 64), nn.ELU(), nn.Dropout(p), nn.Linear(64, o)) for k, o in outs.items()})
    def forward(self, x, mask):
        v = self.conv(x).mean(-1)
        h = v
        if self.context is not None:
            u = self.to_ctx(v)
            U = u.new_zeros(*mask.shape, u.shape[1]); U[mask] = u
            h = torch.cat([v, self.context(U, mask)[mask]], 1)
        out = {k: m(h) for k, m in self.pred.items()}
        out["hr"], out["rmssd"] = out["hr"][:, 0], out["rmssd"][:, 0]
        return out

class PlainFusion(nn.Module):
    def __init__(self, c_in, d=128):
        super().__init__()
        self.pa = nn.Sequential(nn.Conv1d(c_in, d, 1, bias=False), nn.BatchNorm1d(d))
        self.pb = nn.Sequential(nn.Conv1d(c_in, d, 1, bias=False), nn.BatchNorm1d(d))
        self.out = nn.Sequential(nn.Conv1d(2 * d, d, 1, bias=False), nn.BatchNorm1d(d), nn.ReLU(inplace=True))
    def forward(self, fa, fb): return self.out(torch.cat([self.pa(fa), self.pb(fb)], 1))

def build_model(ablate=None):
    if ablate in ("plaincnn", "plaincnn_ctx"): return PlainCNN(context=ablate == "plaincnn_ctx").to(DEV)
    m = M.PMRTTemporal(L=SEQ_LEN, stage_head=True)
    if ablate == "physe": m.branch_eeg, m.branch_emg = PlainResNet(), PlainResNet()
    if ablate == "stdse": m.branch_eeg, m.branch_emg = StdSEResNet(), StdSEResNet()
    if ablate == "singleconv": m.branch_eeg, m.branch_emg = SingleConvNet(), SingleConvNet()
    if ablate == "oneconv":
        m.branch_eeg, m.branch_emg = OneConvNet(), OneConvNet()
        m.fusion = nn.ModuleList([M.CrossModalFusion(64, 128, 16) for _ in range(3)])
    if ablate == "fusion": m.fusion = nn.ModuleList([PlainFusion(c) for c in (128, 256, 512)])
    return m.to(DEV)

for tag, a in (("with module", None), (ABLATE, ABLATE)):
    n = sum(p.numel() for p in build_model(a).parameters())
    print(f"{tag:12s}: {n / 1e6:.2f} M parameters")

In [ ]:
# Data and the same subject-wise folds as the final run
X, idx = M.load_data(str(DATA))
REC = idx.record.to_numpy()
FOLD = json.loads(FOLDS_FILE.read_text())
DIS, STG = idx.dis.to_numpy(np.int64), idx.stg.to_numpy(np.int64)
print("windows:", X.shape, "| subjects:", idx.record.nunique())

In [ ]:
# Train the model without the module: same folds, losses, optimiser and early stopping as the final run
ABL.mkdir(parents=True, exist_ok=True)
(ABL / "config.json").write_text(json.dumps({k: globals()[k] for k in ["N_FOLDS", "SEQ_LEN", "EPOCHS", "PATIENCE", "MIN_EPOCHS", "BATCH_SEQ", "LR", "WD", "PER_SUBJECT", "EMA_DECAY", "SEED"]} | {"ablate": ABLATE}, indent=1))
for k in FOLDS:
    fdir = ABL / f"fold{k}"
    if (fdir / "test_pred.csv").exists():
        print(f"fold {k}: already finished - skipped"); continue
    if time.time() - T_START > TIME_LIMIT_H * 3600:
        print(f"time limit reached before fold {k} - run again to continue"); break
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    rows = M.fold_rows(idx, FOLD, k, N_FOLDS)
    tg = M.Targets(idx, rows["train"])
    yz = tg.encode()
    Y = {"hr": yz[:, 0], "rmssd": yz[:, 1], "dis": DIS, "stg": STG}
    tr = M.SequenceBatches(X, Y, idx, rows["train"], BATCH_SEQ, DEV, SEQ_LEN, PER_SUBJECT, train=True)
    va = M.SequenceBatches(X, Y, idx, rows["val"], 64, DEV, SEQ_LEN, train=False)
    te = M.SequenceBatches(X, Y, idx, rows["test"], 64, DEV, SEQ_LEN, train=False)
    print(f"\n===== fold {k} | train {len(np.unique(REC[rows['train']]))} subj | val {len(np.unique(REC[rows['val']]))} | test {len(np.unique(REC[rows['test']]))} =====")
    loss_fn = M.MultiTaskLoss(idx.iloc[rows["train"]], DEV)
    model = build_model(ABLATE)
    M.train_fold(model, tr, va, loss_fn, tg, DIS, STG, REC, fdir, epochs=EPOCHS, lr=LR, wd=WD,
                 patience=PATIENCE, min_epochs=MIN_EPOCHS, use_aug=True, ema_decay=EMA_DECAY)
    model.load_state_dict(torch.load(fdir / "best_stress.pt", map_location=DEV, weights_only=False)["model"])
    j, p = M.predict(model, te)
    model.load_state_dict(torch.load(fdir / "best_dis.pt", map_location=DEV, weights_only=False)["model"])
    j2, p2 = M.predict(model, te)
    assert (j == j2).all() and (j == np.sort(rows["test"])).all()
    out = idx.iloc[j][["record", "group", "stage", "clipped", "dis", "stg"] + M.REG].reset_index(drop=True)
    out["fold"] = k
    out["hr_pred"], out["rmssd_pred"] = tg.decode(p["hr"], p["rmssd"])
    for c, name in enumerate(M.DISORDERS): out[f"p_{name}"] = p2["dis"][:, c]
    for c, name in enumerate(M.STAGES): out[f"p_{name}"] = p["stg"][:, c]
    out["mu_hr"], out["mu_rmssd"] = np.median(tg.raw[rows["train"]], 0)
    out.to_csv(fdir / "test_pred.csv", index=False)
    (fdir / "last.pt").unlink(missing_ok=True)
    print(f"fold {k} saved")
print("\ntraining finished")

In [ ]:
# Package the trained model without the module and decide which analysis steps can run
done = [k for k in FOLDS if (ABL / f"fold{k}" / "test_pred.csv").exists()]
print("finished folds of the ablated model:", done, "of", FOLDS)
zip_run = shutil.make_archive(str(ROOT / f"ablation_run_{ABLATE}"), "zip", ROOT / "runs", ABL.name)
print("saved", zip_run, f"({Path(zip_run).stat().st_size / 1e6:.0f} MB)")
HAVE_PRED = len(done) == len(FOLDS) and all((MAIN / f"fold{k}" / "test_pred.csv").exists() for k in FOLDS)
HAVE_CKPT = False
print("comparison of metrics and confusion/ROC figures can run:", HAVE_PRED)


In [ ]:
# Test metrics with vs without the module (same subjects, same metric code)
if HAVE_PRED:
    from scipy.stats import wilcoxon
    from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, balanced_accuracy_score, f1_score
    SC, PC = [f"p_{s}" for s in M.STAGES], [f"p_{c}" for c in M.DISORDERS]

    def load_preds(run):
        return pd.concat([pd.read_csv(run / f"fold{k}" / "test_pred.csv") for k in FOLDS], ignore_index=True)

    def zrec(P, c):
        g = P.groupby("record")[c]
        return ((P[c] - g.transform("mean")) / g.transform("std")).fillna(0)

    def evaluate(P):
        P = P.copy()
        P["si_true"], P["si_pred"] = zrec(P, "hr") - zrec(P, "rmssd"), zrec(P, "hr_pred") - zrec(P, "rmssd_pred")
        P["y_stress"] = (P.si_true > 1).astype(int)
        P["stg_pred"] = P[SC].to_numpy().argmax(1)
        S = P.groupby("record").agg(group=("group", "first"), dis=("dis", "first"), **{c: (c, "mean") for c in PC})
        S6 = S[S.dis >= 0]
        hb = (S.group != "Healthy").astype(int)
        r = {"HR MAE (bpm)": np.mean(np.abs(P.hr_pred - P.hr)), "RMSSD MAE (ms)": np.mean(np.abs(P.rmssd_pred - P.rmssd)),
             "HR r": np.corrcoef(P.hr, P.hr_pred)[0, 1], "RMSSD r": np.corrcoef(P.rmssd, P.rmssd_pred)[0, 1],
             "Stress AUC": roc_auc_score(P.y_stress, P.si_pred),
             "Stage accuracy": accuracy_score(P.stg, P.stg_pred), "Stage kappa": cohen_kappa_score(P.stg, P.stg_pred),
             "Stage macro F1": f1_score(P.stg, P.stg_pred, average="macro"),
             "Disorder balanced acc (6 classes, subject level)": balanced_accuracy_score(S6.dis, S6[PC].to_numpy().argmax(1)),
             "Healthy vs disorder AUC": roc_auc_score(hb, 1 - S.p_Healthy)}
        per = P.assign(ok=(P.stg == P.stg_pred), hr_err=np.abs(P.hr_pred - P.hr)).groupby("record").agg(stage_acc=("ok", "mean"), hr_mae=("hr_err", "mean"))
        return {k: float(v) for k, v in r.items()}, per, P, S

    P_with, P_without = load_preds(MAIN), load_preds(ABL)
    P_with = P_with[P_with.fold.isin(FOLDS)]
    r_with, per_with, P_with, S_with = evaluate(P_with)
    r_without, per_without, P_without, S_without = evaluate(P_without)
    tab = pd.DataFrame({"with module": r_with, LABEL[ABLATE]: r_without})
    tab["difference (with - without)"] = tab.iloc[:, 0] - tab.iloc[:, 1]
    print(f"{P_with.record.nunique()} subjects, {len(P_with):,} windows\n")
    print(tab.round(3).to_string())
    for col, name in (("stage_acc", "stage accuracy per subject"), ("hr_mae", "HR MAE per subject")):
        d = per_with[col] - per_without[col]
        print(f"paired Wilcoxon, {name}: mean difference {d.mean():+.4f}, p = {wilcoxon(d).pvalue:.4f}")
    tab.round(4).to_csv(OUT / "metrics_with_vs_without.csv")
else:
    print("skipped: needs the finished run pmrt_mt3_cv5 and all folds of the ablated model")


In [ ]:
# Confusion matrices and ROC curves of the final model (plus ROC with vs without the module)
if HAVE_PRED:
    from matplotlib.colors import LinearSegmentedColormap
    from sklearn.metrics import confusion_matrix, roc_curve
    CMAP = LinearSegmentedColormap.from_list("b", ["#f4f7fa", "#9fb9d6", NAVY])
    P, S = P_with, S_with
    S6 = S[S.dis >= 0]
    sp, dp = P.stg_pred.to_numpy(), S6[PC].to_numpy().argmax(1)
    ys, ps = P.y_stress, (P.si_pred > 1).astype(int)
    hb = (S.group != "Healthy").astype(int); hp = (S[PC].to_numpy().argmax(1) != 0).astype(int)

    def cm_fig(cm, labels, title, name):
        p = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        fig, ax = plt.subplots(figsize=(5.6 if len(labels) > 4 else 4.6,) * 2); ax.imshow(p, cmap=CMAP, vmin=0, vmax=1)
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{p[i, j]:.2f}\n({int(cm[i, j]):,})", ha="center", va="center", fontsize=8, color="white" if p[i, j] > .55 else "#1f2a36")
        ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right"); ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=10); fig.savefig(OUT / f"{name}.png"); plt.show(); plt.close(fig)

    cm_fig(confusion_matrix(P.stg, sp, labels=range(4)), M.STAGES, f"Sleep stage ({len(P):,} windows)\naccuracy {r_with['Stage accuracy']:.3f}, kappa {r_with['Stage kappa']:.3f}", "cm_sleep_stage")
    cm_fig(confusion_matrix(S6.dis, dp, labels=range(6)), M.DISORDERS, f"Disorder, subject level ({len(S6)} subjects)\nbalanced accuracy {r_with['Disorder balanced acc (6 classes, subject level)']:.3f}", "cm_disorder")
    cm_fig(confusion_matrix(ys, ps, labels=[0, 1]), ["no stress", "stress"], f"Stress windows (SI > 1)\nbalanced accuracy {balanced_accuracy_score(ys, ps):.3f}", "cm_stress")
    cm_fig(confusion_matrix(hb, hp, labels=[0, 1]), ["healthy", "disorder"], f"Healthy vs disorder ({len(S)} subjects)\nbalanced accuracy {balanced_accuracy_score(hb, hp):.3f}", "cm_healthy_vs_disorder")

    def roc_fig(curves, title, name):
        fig, ax = plt.subplots(figsize=(5.2, 5.0))
        for lab, y, s, c in curves:
            f, t, _ = roc_curve(y, s); k = np.linspace(0, len(f) - 1, min(len(f), 600)).astype(int)
            ax.plot(f[k], t[k], color=c, lw=2, label=f"{lab} (AUC {roc_auc_score(y, s):.3f})")
        ax.plot([0, 1], [0, 1], "k:", lw=1); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.set_aspect("equal")
        ax.set_title(title, fontsize=10); ax.legend(fontsize=8, loc="lower right"); fig.savefig(OUT / f"{name}.png"); plt.show(); plt.close(fig)

    roc_fig([("stress index", ys, P.si_pred, NAVY)], f"Stress detection ({len(P):,} windows)", "roc_stress")
    roc_fig([("healthy vs disorder", hb, 1 - S.p_Healthy, NAVY)], f"Healthy vs disorder ({len(S)} subjects)", "roc_healthy_vs_disorder")
    roc_fig([(s, (P.stg == i).astype(int), P[f"p_{s}"], CLS[i]) for i, s in enumerate(M.STAGES)], "Sleep stage, one class vs rest (windows)", "roc_sleep_stage_per_class")
    roc_fig([(c, (S6.dis == i).astype(int), S6[f"p_{c}"], CLS[i]) for i, c in enumerate(M.DISORDERS)], f"Disorder, one class vs rest ({len(S6)} subjects)", "roc_disorder_per_class")
    Q, SQ = P_without, S_without
    roc_fig([("with module", ys, P.si_pred, NAVY), ("without module", Q.y_stress, Q.si_pred, SLATE)], "Stress detection, with vs without the module", "roc_stress_with_vs_without")
    roc_fig([("with module", hb, 1 - S.p_Healthy, NAVY), ("without module", (SQ.group != "Healthy").astype(int), 1 - SQ.p_Healthy, SLATE)], "Healthy vs disorder, with vs without the module", "roc_healthy_vs_disorder_with_vs_without")
else:
    print("skipped: needs the finished run pmrt_mt3_cv5 and all folds of the ablated model")


In [ ]:
# Save numbers, build the zip of figures and tables
if HAVE_PRED:
    summary = {"ablate": ABLATE, "folds": FOLDS, "metrics": tab.round(4).to_dict(),
               "stage_auc_per_class": {s: roc_auc_score((P.stg == i).astype(int), P[f"p_{s}"]) for i, s in enumerate(M.STAGES)},
               "disorder_auc_per_class": {c: roc_auc_score((S6.dis == i).astype(int), S6[f"p_{c}"]) for i, c in enumerate(M.DISORDERS) if (S6.dis == i).any()},
               "cm_stage": confusion_matrix(P.stg, sp, labels=range(4)).tolist(), "cm_disorder": confusion_matrix(S6.dis, dp, labels=range(6)).tolist(),
               "cm_stress": confusion_matrix(ys, ps, labels=[0, 1]).tolist(), "cm_healthy_vs_disorder": confusion_matrix(hb, hp, labels=[0, 1]).tolist()}
    if HAVE_CKPT: summary["separability"] = SEP.round(4).to_dict("records")
    (OUT / "report_numbers.json").write_text(json.dumps(summary, indent=1, default=float))
zip_path = shutil.make_archive(str(ROOT / f"ablation_analysis_{ABLATE}"), "zip", OUT)
print("saved", zip_path, "|", len(list(OUT.glob("*.png"))), "figures")
print("download:", Path(zip_run).name, "(trained model without the module) and", Path(zip_path).name, "(figures and tables)")
